# Systematic CpG probe removal: effect on clock performance

Investigates what happens when entire CpG probes are **systematically absent
from every sample** — simulating whole-assay failures, QC-failed probe batches,
or array technologies that do not cover a genomic region.

This differs from the random masking in notebook 03 in two key ways:

- **Same probes are removed from every sample** at each level, not a random
  per-sample draw.
- **Removal is ordered by a strategy** (variance, mean methylation, or vocab
  position), so the degradation curve reflects what happens when you lose the
  least — or most — informative probes first.

An overlay of the random masking curve from notebook 03 is included so you can
directly compare: if systematic removal hurts more than random masking at the
same fraction, those specific probes carry disproportionate signal.

Helper functions live in `eval_utils.py`.

## Parameters
Set everything here, then run all cells.

In [1]:
from pathlib import Path
import numpy as np

# ── Which clocks ──────────────────────────────────────────────────
# None = "age" + every best_head_<clock>.pt found in HEAD_PATH
CLOCKS     = None
EVAL_SPLIT = "all"    # split to evaluate; "all" uses every sample

# ── Paths ─────────────────────────────────────────────────────────
ROOT          = Path("/work/gr-fe/bryan/EpiHydra")
DATA_PATH     = ROOT / "data/03_final/combined/methylation.pkl"
MODEL_PATH    = ROOT / "data/03_final/methylgpt/medium"
EPIHYDRA_PATH = ROOT / "results/tensorboard/checkpoints/nlayers6_nhead4_emb256_lr1e-04_bs32_mask0.3_heads4_20260902_105750"
HEAD_PATH     = ROOT / "results/tensorboard/ckpt/clocks4_nlayers6_nhead4_emb256_lora_lr1e-04_bs32_mask0.15_heads4_20261105_072540"
LORA_CKPT     = EPIHYDRA_PATH / "best_model.pt"

# ── Model settings (must match training) ──────────────────────────
NUM_HEADS  = 4
LORA_R     = 16
LORA_ALPHA = 8

# ── Data split ────────────────────────────────────────────────────
SPLIT_RATIOS = dict(train_ratio=0.98, valid_ratio=0.01, test_ratio=0.01)
BATCH_SIZE   = 16
RANDOM_STATE = 42

# ── Analysis 1: synthetic masking curve ───────────────────────────
# Fractions of CpGs to additionally mask on top of real missingness
MASK_FRACTIONS = np.round(np.arange(0.0, 1.0, 0.1), 2).tolist()
MASK_SEED      = 42    # for reproducibility across fractions

# ── Analysis 2: real coverage bins ────────────────────────────────
N_COVERAGE_BINS = 5    # number of equal-width bins from 0 % to 100 %
MIN_BIN_SAMPLES = 3    # bins with fewer samples are dropped

# ── Output ────────────────────────────────────────────────────────
SAVE_DIR = None        # Path to save figures, or None

# ── Analysis: systematic probe removal ───────────────────────────────────────
# Removal order strategy: "variance" (least informative first),
# "mean" (most extreme methylation first), or "sequential" (vocab order)
REMOVAL_STRATEGY  = "variance"
# Fractions of vocab CpGs to remove at each step
REMOVAL_FRACTIONS = MASK_FRACTIONS   # reuse the same levels as notebook 03

# Whether to load a pre-run random masking result for overlay comparison.
# Set to a path string (e.g. "mask_results.csv") or None to skip the overlay.
RANDOM_RESULTS_CSV = None

## Imports

In [2]:
import json
import pickle
import warnings

warnings.filterwarnings("ignore", message=".*IProgress.*")
warnings.filterwarnings("ignore", message=".*flash_attn.*")
warnings.filterwarnings("ignore", message=".*nested tensor.*")

import matplotlib.pyplot as plt
import pandas as pd
import torch

%matplotlib inline

from methylgpt import MethylGPTModel, MethylVocab
from scgpt.tokenizer import tokenize_and_pad_batch, random_mask_value
import eval_utils as eu
from epihydra import VocabWrapper
from train import build_multiclock_dataloaders

device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
clock_names = eu.resolve_clocks(CLOCKS, HEAD_PATH)

print(f"Using device : {device}")
print(f"Clocks       : {clock_names}")
print(f"Split        : {EVAL_SPLIT}")

Using device : cuda
Clocks       : ['age', 'grimageadapter_age', 'hannum_age', 'horvath_age', 'phenoage_age']
Split        : all


## Load data, config and vocab

In [3]:
print("Loading data...")
with open(DATA_PATH, "rb") as f:
    data = pickle.load(f)

with open(MODEL_PATH / "args.json") as f:
    pretrain_config = json.load(f)

vocab = VocabWrapper(MODEL_PATH / "vocab.json", pretrain_config["n_hvg"])

Loading data...


## Build dataloaders
Saved training scalers are reused where available. Any clock without one (typically `"age"`) gets a fresh scaler fitted on this training split.

In [13]:
saved_scalers = eu.load_saved_scalers(HEAD_PATH)

splits = build_multiclock_dataloaders(
    data         = data,
    vocab        = vocab,
    clock_names  = clock_names,
    batch_size   = BATCH_SIZE,
    num_workers  = 0,
    random_state = RANDOM_STATE,
    verbose      = True,
    scalers      = saved_scalers,
    **SPLIT_RATIOS,
)
scalers = splits["scalers"]

Samples in meta : 1743
Samples in CpG  : 1743
Common samples  : 1743
Samples after drop : 1743
  age                 : 1743 labeled
  grimageadapter_age  : 1743 labeled
  hannum_age          : 1743 labeled
  horvath_age         : 1743 labeled
  phenoage_age        : 1743 labeled

Vocab CpGs : 49,159 | Present : 49,156 (100.0%) | Missing : 3

Train : 1708 | Valid : 17 | Test : 18
ℹ  age                 : scaler fitted on training values.
ℹ  grimageadapter_age  : reusing supplied pre-fitted scaler.
ℹ  hannum_age          : reusing supplied pre-fitted scaler.
ℹ  horvath_age         : reusing supplied pre-fitted scaler.
ℹ  phenoage_age        : reusing supplied pre-fitted scaler.
✓ Multi-clock data loaded and ready.



## Build the multi-clock model
Frozen, LoRA-merged backbone + one trained head per clock.

In [14]:
model = eu.build_multiclock_model(
    MODEL_PATH, vocab, pretrain_config,
    lora_ckpt   = LORA_CKPT,
    head_path   = HEAD_PATH,
    clock_names = clock_names,
    device      = device,
    num_heads   = NUM_HEADS,
    lora_r      = LORA_R,
    lora_alpha  = LORA_ALPHA,
)

/work/gr-fe/bryan/miniconda3/envs/methylgpt/lib/python3.11/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer was not TransformerEncoderLayer
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


Loaded all params from /work/gr-fe/bryan/EpiHydra/data/03_final/methylgpt/medium/medium-best_model_epoch6.pt

Loading frozen LoRA adapters from /work/gr-fe/bryan/EpiHydra/results/tensorboard/checkpoints/nlayers6_nhead4_emb256_lr1e-04_bs32_mask0.3_heads4_20260902_105750/best_model.pt
  source run   : nlayers6_nhead4_emb256_lr1e-04_bs32_mask0.3_heads4_20260902_105750
  source epoch : 85  (valid MAE 4.864)
  lora_r=16 lora_alpha=8 targets=['Wqkv', 'out_proj', 'linear1', 'linear2']
  adapter tensors in checkpoint : 52
  adapter tensors in model      : 52 (lora_A + lora_B pairs)
  lora_B with non-zero weights  : 26/26
  adapters merged into base weights

  loaded head age                  <- best_model.pt
  loaded head grimageadapter_age   <- best_head_grimageadapter_age.pt (epoch 46)
  loaded head hannum_age           <- best_head_hannum_age.pt (epoch 47)
  loaded head horvath_age          <- best_head_horvath_age.pt (epoch 31)
  loaded head phenoage_age         <- best_head_phenoage_age.p

## Baseline predictions
Establishes unmasked performance before any probes are removed. The loader is rebuilt unshuffled so rows stay aligned to sample IDs.

In [6]:
split_names = ["train", "valid", "test"] if EVAL_SPLIT == "all" else [EVAL_SPLIT]

tables = []
for split in split_names:
    loader = eu.ordered_loader(splits[f"{split}_loader"])
    ids    = splits[f"{split}_df"].index
    table  = eu.predict_multiclock(model, loader, scalers, device, clock_names, ids)
    table.insert(0, "split", split)
    tables.append(table)

predictions = pd.concat(tables)
print(f"{len(predictions)} samples x {len(clock_names)} clocks")

1743 samples x 5 clocks


In [7]:
# Baseline per-clock metrics (pred vs. true for the same clock only)
baseline_rows = []
for name in clock_names:
    sub = predictions[[f"{name}_pred", f"{name}_true"]].dropna()
    p, t = sub.iloc[:, 0].to_numpy(), sub.iloc[:, 1].to_numpy()
    if len(p) < 2:
        continue
    from scipy.stats import pearsonr
    from sklearn.metrics import r2_score
    baseline_rows.append({
        "clock"    : name,
        "n"        : len(p),
        "mae"      : round(float(np.mean(np.abs(p - t))), 3),
        "pearson_r": round(float(pearsonr(p, t)[0]), 3),
        "r2"       : round(float(r2_score(t, p)), 3),
    })

baseline = pd.DataFrame(baseline_rows).set_index("clock")
print("Baseline performance (0 % extra masking)")
display(baseline)

Baseline performance (0 % extra masking)


,n,mae,pearson_r,r2
clock,,,,
age,1743,4.739,0.969,0.937
grimageadapter_age,1743,4.126,0.937,0.877
hannum_age,1743,3.152,0.921,0.848
horvath_age,1743,3.606,0.977,0.955
phenoage_age,1743,4.946,0.975,0.950


## Build unshuffled evaluation loaders

In [8]:
# Rebuild loaders without shuffling so probe-removal results are
# row-aligned across fractions and comparable to the baseline above.
eval_loaders = []
for split in split_names:
    eval_loaders.append(eu.ordered_loader(splits[f"{split}_loader"]))

## Systematic probe removal

CpGs are ranked once on the training split by the chosen strategy, then removed in that fixed order from every sample simultaneously at each fraction level.

In [15]:
def rank_cpgs(splits, strategy="variance", split="train"):
    """
    Return a vocab-ordered array of CpG token indices, ranked by a chosen
    strategy so we know which probes to remove first in systematic removal.
 
    Parameters
    ----------
    splits   : dict from build_multiclock_dataloaders
    strategy : "variance"   — remove lowest-variance probes first
                              (least informative across the cohort)
               "mean"       — remove probes closest to the population mean
                              (least discriminating by absolute value)
               "sequential" — remove in vocab order (position baseline;
                              useful to confirm the model has no positional
                              artefacts and that the effect is probe-driven,
                              not positional)
    split    : which split's data to compute statistics over (default "train")
 
    Returns
    -------
    ranked_indices : np.ndarray of int — vocab positions, ordered so that
                     index 0 is removed first
    strategy_label : str for plot titles / legends
    """
    df    = splits[f"{split}_df"]
    # Stack all CpG arrays into (n_samples, n_cpgs)
    X     = np.stack(df["data"].to_numpy())   # may contain NaN
    n_cpg = X.shape[1]
 
    if strategy == "sequential":
        return np.arange(n_cpg), "sequential (vocab order)"
 
    with np.errstate(all="ignore"):
        stat = np.nanvar(X, axis=0) if strategy == "variance" else np.nanmean(np.abs(X), axis=0)
 
    # NaN probes (never observed) go last — they're already missing so removing
    # them has no effect; placing them last keeps the curve honest
    stat = np.where(np.isfinite(stat), stat, -np.inf if strategy == "variance" else np.inf)
 
    if strategy == "variance":
        ranked = np.argsort(stat)          # ascending: lowest variance first
        label  = "ascending variance (least informative first)"
    else:
        ranked = np.argsort(stat)[::-1]   # descending: highest mean first
        label  = "descending mean value (most extreme first)"
 
    return ranked.astype(int), label

In [16]:
ranked_indices, strategy_label = rank_cpgs(
    splits,
    strategy = REMOVAL_STRATEGY,
    split    = "train",
)
print(f"Removal strategy : {strategy_label}")
print(f"Total CpGs ranked: {len(ranked_indices):,}")
print(f"First 5 to remove: vocab positions {ranked_indices[:5]}")

Removal strategy : ascending variance (least informative first)
Total CpGs ranked: 49,159
First 5 to remove: vocab positions [27528 27505 27515 27518 27519]


/tmp/4828274/ipykernel_3548610/3812448322.py:34: RuntimeWarning: Degrees of freedom <= 0 for slice.
  stat = np.nanvar(X, axis=0) if strategy == "variance" else np.nanmean(np.abs(X), axis=0)


In [18]:
@torch.no_grad()
def evaluate_systematic_removal(model, loader, scalers, device, clock_names,
                                 ranked_indices, removal_fractions, pad_id,
                                 mask_value=-1.0):
    """
    Remove CpG probes SYSTEMATICALLY across all samples simultaneously,
    simulating a whole assay or genomic-region failure.
 
    At each removal fraction f, the first int(f * n_cpg) probes in
    `ranked_indices` are set to `mask_value` for EVERY sample in every batch.
    This is different from random masking in two key ways:
 
      * The SAME probes are removed from every sample (assay-level failure).
      * The order in which probes are removed is determined by `ranked_indices`
        (e.g. lowest-variance probes first), so the curve reflects what
        happens when you progressively lose the least — or most — informative
        probes first.
 
    Parameters
    ----------
    model              : MultiClockPredictor, eval mode
    loader             : unshuffled DataLoader
    scalers            : dict[clock, scaler]
    device             : torch.device
    clock_names        : list[str]
    ranked_indices     : np.ndarray — vocab positions in removal order
                         (index 0 is removed first); from rank_cpgs()
    removal_fractions  : list[float] — e.g. [0.0, 0.1, ..., 0.9]
    pad_id             : int — model.pad_id
    mask_value         : float — value to inject (default -1, same as training)
 
    Returns
    -------
    DataFrame with columns [clock, removal_frac, n_removed, mae, pearson_r, r2, n]
    """
    import pandas as pd
 
    n_cpg = len(ranked_indices)
    rows  = []
 
    for frac in removal_fractions:
        n_remove    = int(frac * n_cpg)
        probes_off  = set(ranked_indices[:n_remove].tolist())   # vocab positions to blank
 
        all_preds  = {name: [] for name in clock_names}
        all_labels = {name: [] for name in clock_names}
 
        for gene_ids, _, target_vals, ages, valid, _ in loader:
            gene_ids    = gene_ids.to(device)
            target_vals = target_vals.to(device)
 
            # Build a mask for positions whose vocab index is in probes_off.
            # gene_ids holds token IDs (integers), which are exactly the vocab
            # positions, so we can compare directly.
            if probes_off:
                # Boolean tensor: True where this token is a removed probe
                remove_mask = torch.zeros_like(gene_ids, dtype=torch.bool)
                for pos in probes_off:
                    remove_mask |= gene_ids.eq(pos)
                vals_masked = target_vals.clone()
                vals_masked[remove_mask] = mask_value
            else:
                vals_masked = target_vals
 
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                out = model(gene_ids, vals_masked, clock_names=clock_names)
 
            for name in clock_names:
                v = valid[name].numpy()
                if v.sum() == 0:
                    continue
                pred_norm = out[name].float().cpu().numpy()[v].reshape(-1, 1)
                pred_age  = scalers[name].inverse_transform(pred_norm).ravel()
                all_preds[name].extend(pred_age)
                all_labels[name].extend(ages[name].numpy()[v])
 
        for name in clock_names:
            p = np.array(all_preds[name])
            t = np.array(all_labels[name])
            n = len(p)
            if n < 2:
                rows.append(dict(clock=name, removal_frac=frac, n_removed=n_remove,
                                 mae=np.nan, pearson_r=np.nan, r2=np.nan, n=n))
                continue
            mae = float(np.mean(np.abs(p - t)))
            r   = float(pearsonr(p, t)[0]) if np.std(p) > 0 and np.std(t) > 0 else np.nan
            r2  = float(r2_score(t, p))    if np.std(p) > 0 else np.nan
            rows.append(dict(clock=name, removal_frac=frac, n_removed=n_remove,
                             mae=mae, pearson_r=r, r2=r2, n=n))
 
    return pd.DataFrame(rows)

In [ ]:
systematic_results_parts = []
for split, loader in zip(split_names, eval_loaders):
    res = evaluate_systematic_removal(
        model, loader, scalers, device,
        clock_names       = clock_names,
        ranked_indices    = ranked_indices,
        removal_fractions = REMOVAL_FRACTIONS,
        pad_id            = model.pad_id,
    )
    res.insert(0, "split", split)
    systematic_results_parts.append(res)

systematic_results = pd.concat(systematic_results_parts, ignore_index=True)
print(systematic_results.round(3).to_string(index=False))

In [ ]:
# Optionally load a pre-computed random masking result for overlay comparison
random_results = None
if RANDOM_RESULTS_CSV is not None:
    random_results = pd.read_csv(RANDOM_RESULTS_CSV)
    print(f"Loaded random masking results from {RANDOM_RESULTS_CSV}")

for metric in ("mae", "pearson_r", "r2"):
    fig, axes = eu.plot_systematic_removal_curves(
        systematic_results,
        random_results = random_results,
        metric         = metric,
        clock_names    = clock_names,
        strategy_label = REMOVAL_STRATEGY,
    )
    if SAVE_DIR:
        fig.savefig(Path(SAVE_DIR) / f"systematic_removal_{metric}.pdf")
        fig.savefig(Path(SAVE_DIR) / f"systematic_removal_{metric}.png")
    plt.show()

## Removal strategy comparison

Runs all three strategies and overlays the degradation curves per clock. If `"variance"` and `"sequential"` produce identical curves the effect is driven by quantity of missing data alone. If `"variance"` degrades more slowly, the model is robust to losing low-information probes and relies on a smaller set of high-variance ones.

In [ ]:
COMPARE_METRIC = "mae"

strategy_dfs = {}
for strat in ("variance", "sequential", "mean"):
    r_idx, r_label = eu.rank_cpgs(splits, strategy=strat, split="train")
    parts = []
    for split, loader in zip(split_names, eval_loaders):
        res = eu.evaluate_systematic_removal(
            model, loader, scalers, device,
            clock_names       = clock_names,
            ranked_indices    = r_idx,
            removal_fractions = REMOVAL_FRACTIONS,
            pad_id            = model.pad_id,
        )
        res.insert(0, "split", split)
        parts.append(res)
    strategy_dfs[strat] = (pd.concat(parts, ignore_index=True), r_label)

with plt.rc_context(eu._NM_RC):
    n      = len(clock_names)
    fig_w  = min(183, 55 * n) / 25.4
    fig, axes = plt.subplots(1, n, figsize=(fig_w, 65 / 25.4),
                             sharey=True, constrained_layout=True)
    if n == 1:
        axes = [axes]

    colors  = {"variance": "#2166AC", "sequential": "#4DAC26", "mean": "#D6604D"}
    linesty = {"variance": "-",       "sequential": "--",      "mean": "-."}

    for ax, name in zip(axes, clock_names):
        for strat, (df, label) in strategy_dfs.items():
            sub = df[df["clock"] == name].sort_values("removal_frac")
            ax.plot(sub["removal_frac"] * 100, sub[COMPARE_METRIC],
                    color=colors[strat], ls=linesty[strat], lw=1.0,
                    marker="o", markersize=2.5, label=strat)
        ax.set_title(eu._clean(name), pad=4, fontweight="bold")
        ax.set_xlabel("CpGs removed (%)")
        ax.set_ylim(bottom=0)
        ax.legend(frameon=False, fontsize=5)

    ylabel = {"mae": "MAE (years)", "pearson_r": "Pearson r", "r2": "R²"}[COMPARE_METRIC]
    axes[0].set_ylabel(ylabel)
    fig.suptitle(f"Removal strategy comparison — {COMPARE_METRIC}",
                 fontsize=8, fontweight="bold")

if SAVE_DIR:
    fig.savefig(Path(SAVE_DIR) / f"strategy_comparison_{COMPARE_METRIC}.pdf")
    fig.savefig(Path(SAVE_DIR) / f"strategy_comparison_{COMPARE_METRIC}.png")
plt.show()

## Summary: baseline vs. worst removal

Compare unmasked baseline to performance at the highest removal fraction, for the chosen strategy.

In [ ]:
worst_frac = max(REMOVAL_FRACTIONS)
worst_syst = systematic_results[
    systematic_results["removal_frac"] == worst_frac
].set_index("clock")

summary = baseline[["mae", "pearson_r", "r2"]].copy()
summary.columns = [f"baseline_{c}" for c in summary.columns]
for metric in ("mae", "pearson_r", "r2"):
    summary[f"syst_{metric}"]        = worst_syst[metric]
    summary[f"delta_syst_{metric}"]  = (
        summary[f"syst_{metric}"] - summary[f"baseline_{metric}"]
    ).round(3)

print(f"Strategy: {REMOVAL_STRATEGY} | worst fraction: {worst_frac:.0%}")
display(summary.round(3))